# ANA-070 — numerical and provenance validation

This notebook checks the recorded profile-prior experiment and synthetic control without repeating either optimization. It supports the comparison figures in `analysis.ipynb`; it does not set scientific pass/fail thresholds for biological resemblance.

The checks cover output/source/input hashes, paired genotype masks and cube geometries, identical empirical descriptors, all 24 full normalized interaction fields, 36 independently recomputed calibration objectives, the original search budgets and 4,326 independently reconstructed WHT summaries. The WHT calculation uses an independent Hadamard transform. Objective checks use the frozen calibration arrays and observation procedure. Full methods are in `audit_tools.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AIM.md').exists())
ANA = ROOT / 'analyses/2026-11-19_ANA-070_run-178-trpb-profile-prior-versus-synthetic-control'
sys.path.insert(0, str(ANA / 'notebooks'))
import audit_tools

In [2]:
audit = audit_tools.verify()
display(pd.Series(audit, name='Verified record').to_frame())

,Verified record
scope,TrpB RUN-178 profile versus archived RUN-175 s...
provenance,"[{'run': 'RUN-175', 'output_hashes': 139, 'sou..."
frozen_control_hashes_verified,6
profile_input_hashes_verified,10
measured_genotypes,159129
paired_seeds,24
paired_full_field_nodes,3840000
maximum_normalized_g_difference,0.0
unchanged_computational_methods,"[__init__, _sequences, latent_components, read..."
empirical_graph_and_cube_rows_exact,True


## Calibration and pairing checks

Every selected objective is recomputed from its saved parameters. Endpoint calibrations use the original RUN-172 algorithm, while intermediate searches retain RUN-175's budget and consider the two newly fitted profile endpoints. All searches reached their iteration cap; successful reproduction does not establish optimizer convergence or an attainable model optimum.

The frozen synthetic control is read, not recalibrated. Matching interaction fields confirms pairing for this comparison; profile additive values are expected to differ. Small cross-platform differences are quantified rather than rounded away in the audit tables.

In [3]:
for name in ['fit_audit.csv', 'fit_trace_audit.csv', 'paired_field_audit.csv']:
    frame = pd.read_csv(ANA / 'tables' / name)
    print(name, len(frame), 'rows')
    display(frame.head(6))

fit_audit.csv 36 rows


,run,observation,ratio,objective_recomputed,maximum_component_difference,optimizer_converged,selected_source,objective_quantile,objective_mean_sd,objective_floor_cdf,objective_total
0,RUN-175,none,0.0,0.113967,0.0,False,frozen_RUN172_anchor,0.076879,0.034097,0.002991,0.113967
1,RUN-175,none,0.5,0.086099,0.0,False,differential_evolution,0.058554,0.025527,0.002018,0.086099
2,RUN-175,none,1.0,0.072260,0.0,False,differential_evolution,0.053361,0.016793,0.002107,0.072260
3,RUN-175,none,2.0,0.058594,0.0,False,differential_evolution,0.047292,0.009539,0.001763,0.058594
4,RUN-175,none,4.0,0.043608,0.0,False,frozen_RUN172_anchor,0.036522,0.006441,0.000645,0.043608
5,RUN-175,none,8.0,0.035166,0.0,False,differential_evolution,0.030227,0.004434,0.000504,0.035166


fit_trace_audit.csv 42 rows


,run,phase,trace,evaluations,expected_evaluations,selected_objective,minimum_trace_objective,optimizer_converged
0,RUN-175,selected,TrpB_none_r0.csv.gz,1,1,0.113967,0.113967,False
1,RUN-175,selected,TrpB_none_r0.5.csv.gz,1954,1954,0.086099,0.086099,False
2,RUN-175,selected,TrpB_none_r1.csv.gz,1954,1954,0.072260,0.072260,False
3,RUN-175,selected,TrpB_none_r2.csv.gz,1954,1954,0.058594,0.058594,False
4,RUN-175,selected,TrpB_none_r4.csv.gz,1,1,0.043608,0.043608,False
5,RUN-175,selected,TrpB_none_r8.csv.gz,1954,1954,0.035166,0.035166,False


paired_field_audit.csv 24 rows


,seed,cohort,nodes,maximum_g_difference,additive_rms_difference
0,57117794,reused,160000,3.996803e-15,1.990644
1,57117795,reused,160000,3.330669e-15,2.809543
2,57117796,reused,160000,3.996803e-15,1.816588
3,57117797,reused,160000,3.441691e-15,2.153095
4,57117798,reused,160000,3.552714e-15,2.026791
5,57117799,reused,160000,3.552714e-15,4.516286


## What a passing audit means

These checks establish that the figures compare complete, internally consistent outputs under the recorded protocol. They do not validate the biological observation model, prove broad superiority of the profile prior, or turn eight generator seeds into independent biological replication. The scientific interpretation remains the mixed comparison reported in `analysis.ipynb` and `analysis.md`.